In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

# Initialize Spark session
spark = SparkSession.builder \
    .appName("HandlingNullDataDemo") \
    .getOrCreate()

In [2]:
# Create DataFrame with null values
data = [
    (1, "John", 28, "New York"),
    (2, "Emily", None, "Los Angeles"),
    (3, "Michael", 35, None),
    (4, "Sarah", None, "Houston"),
    (5, "David", 40, None),
    (6, None, None, None),
    (7, "Alice", None, None)
]

columns = ["id", "name", "age", "city"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame with Null Values:")
df.show()

Original DataFrame with Null Values:
+---+-------+----+-----------+
| id|   name| age|       city|
+---+-------+----+-----------+
|  1|   John|  28|   New York|
|  2|  Emily|NULL|Los Angeles|
|  3|Michael|  35|       NULL|
|  4|  Sarah|NULL|    Houston|
|  5|  David|  40|       NULL|
|  6|   NULL|NULL|       NULL|
|  7|  Alice|NULL|       NULL|
+---+-------+----+-----------+



In [3]:
# 1. Detecting null values
print("Detect Null Values in Each Column:")
df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

Detect Null Values in Each Column:
+---+----+---+----+
| id|name|age|city|
+---+----+---+----+
|  0|   1|  4|   4|
+---+----+---+----+



In [4]:
# 2. Filter rows where a specific column is null
print("Filter Rows Where 'age' is Null:")
df.filter(col("age").isNull()).show()

Filter Rows Where 'age' is Null:
+---+-----+----+-----------+
| id| name| age|       city|
+---+-----+----+-----------+
|  2|Emily|NULL|Los Angeles|
|  4|Sarah|NULL|    Houston|
|  6| NULL|NULL|       NULL|
|  7|Alice|NULL|       NULL|
+---+-----+----+-----------+



In [5]:
# 3. Filter rows where no null values exist
print("Filter Rows with No Null Values:")
df.filter(~col("age").isNull() & ~col("name").isNull() & ~col("city").isNull()).show()

Filter Rows with No Null Values:
+---+----+---+--------+
| id|name|age|    city|
+---+----+---+--------+
|  1|John| 28|New York|
+---+----+---+--------+



In [6]:
# 4. Replace null values with default values
print("Replace Null Values with Defaults:")
df_fill_defaults = df.fillna({"name": "Unknown", "age": 0, "city": "Unknown City"})
df_fill_defaults.show()

Replace Null Values with Defaults:
+---+-------+---+------------+
| id|   name|age|        city|
+---+-------+---+------------+
|  1|   John| 28|    New York|
|  2|  Emily|  0| Los Angeles|
|  3|Michael| 35|Unknown City|
|  4|  Sarah|  0|     Houston|
|  5|  David| 40|Unknown City|
|  6|Unknown|  0|Unknown City|
|  7|  Alice|  0|Unknown City|
+---+-------+---+------------+



In [7]:
# 5. Replace null values in a single column with a calculated value
average_age = df.selectExpr("avg(age)").collect()[0][0]
print(f"Replace Null Values in 'age' with Average Age ({average_age}):")
df_fill_age = df.fillna({"age": average_age})
df_fill_age.show()

Replace Null Values in 'age' with Average Age (34.333333333333336):
+---+-------+---+-----------+
| id|   name|age|       city|
+---+-------+---+-----------+
|  1|   John| 28|   New York|
|  2|  Emily| 34|Los Angeles|
|  3|Michael| 35|       NULL|
|  4|  Sarah| 34|    Houston|
|  5|  David| 40|       NULL|
|  6|   NULL| 34|       NULL|
|  7|  Alice| 34|       NULL|
+---+-------+---+-----------+



In [8]:
# 6. Drop rows where specific columns are null
print("Drop Rows Where 'name' or 'city' is Null:")
df_drop_specific = df.dropna(subset=["name", "city"])
df_drop_specific.show()

Drop Rows Where 'name' or 'city' is Null:
+---+-----+----+-----------+
| id| name| age|       city|
+---+-----+----+-----------+
|  1| John|  28|   New York|
|  2|Emily|NULL|Los Angeles|
|  4|Sarah|NULL|    Houston|
+---+-----+----+-----------+

